# Late Fusion Bill Transfer Classifier

PyTorch notebook for binary bill passage transfer experiments using a late-fusion architecture that combines:
- a BERT text encoder for bill text
- a small set of transferable metadata features
- a linear projection for numeric metadata

The main question is whether a model trained on one country transfers to the other:
- train on Canadian bills, test on Canadian and US bills
- train on US bills, test on US and Canadian bills

Feature design defaults to intro-only and transfer-safe metadata to reduce leakage.

Planned outputs:
- country-specific train/test splits
- trained Canada-only and US-only late-fusion models
- a metrics table with PR-AUC, ROC-AUC, F1, precision, recall, accuracy, and balanced accuracy
- token and metadata feature importance summaries

## 0. Setup and Dependencies

All imports live here so the rest of the notebook can stay focused on the transfer experiments.

In [ ]:
import json
import re
import subprocess
import zipfile
from dataclasses import dataclass
from pathlib import Path
from xml.etree import ElementTree as ET

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch import nn
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

ROOT = Path('..').resolve()
DATA = ROOT / 'data'
NORM = DATA / 'normalized'
CA_TXT = DATA / 'canada_bill_text'
US_TXT = DATA / 'us_bill_text'

MODEL_NAME = 'nlpaueb/legal-bert-base-uncased'
MAX_LENGTH = 256
CHUNK_STRIDE = 64
MAX_CHUNKS = 3
BATCH_SIZE = 2
EPOCHS = 2
LR = 2e-4
SEED = 42
TRAIN_FRAC = 0.7
VAL_FRAC = 0.1
TEST_FRAC = 0.2

# Transfer-safe intro-only metadata by default.
NUMERIC_COLS = [
    'year',
    'title_word_count',
    'description_word_count',
    'month_introduced',
]
CATEGORICAL_COLS = [
    'bill_type',
    'chamber',
]
cat_cardinalities = []
TARGET_COL = 'passed'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'ROOT: {ROOT}')
print(f'MODEL_NAME: {MODEL_NAME}')
print(f'DEVICE: {DEVICE}')
print(f'MAX_LENGTH={MAX_LENGTH}, CHUNK_STRIDE={CHUNK_STRIDE}, MAX_CHUNKS={MAX_CHUNKS}, BATCH_SIZE={BATCH_SIZE}')

ROOT: C:\Users\tamze\OneDrive\Documents\GitHub\cpsc440
MODEL_NAME: nlpaueb/legal-bert-base-uncased
Memory-safe defaults: MAX_LENGTH=256, CHUNK_STRIDE=64, MAX_CHUNKS=3, BATCH_SIZE=2


## 1. Load and Merge the Bill Table

Load the normalized bill table, attach the raw text column for Canada and US bills, and keep the text extraction code centralized in this section.

In [ ]:
bills_path = NORM / 'bills.csv'
if not bills_path.exists():
    print('bills.csv not found - running normalize.py ...')
    result = subprocess.run(['python', str(ROOT / 'scripts' / 'normalize.py')], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('normalize.py failed')

bills = pd.read_csv(bills_path, low_memory=False)
bills[TARGET_COL] = pd.to_numeric(bills[TARGET_COL], errors='coerce').fillna(0).astype(int)
bills['introduced_date'] = pd.to_datetime(bills.get('introduced_date'), errors='coerce')
bills['_row_id'] = np.arange(len(bills), dtype=int)

def _ca_xml_text(el) -> str:
    if el.tag in {'Identification', 'Label', 'MarginalNote'}:
        return ''
    if el.tag == 'AmendedText' and el.get('{http://www.w3.org/XML/1998/namespace}lang') == 'fr':
        return ''
    parts = []
    if el.tag == 'Text' and el.text:
        parts.append(el.text.strip())
    for child in el:
        parts.append(_ca_xml_text(child))
        if child.tail:
            parts.append(child.tail.strip())
    return ' '.join(p for p in parts if p)

def load_canada_texts(ca_txt_dir: Path) -> dict:
    texts = {}
    for session_dir in sorted(ca_txt_dir.iterdir()):
        if not session_dir.is_dir():
            continue
        session = session_dir.name
        for txt_file in session_dir.glob('*_english.txt'):
            bill_num = txt_file.name.replace('.pdf_english.txt', '').replace('_english.txt', '')
            texts[f'{session}/{bill_num}'] = txt_file.read_text(encoding='utf-8', errors='replace')
        for xml_file in session_dir.glob('*.xml'):
            key = f'{session}/{xml_file.stem}'
            if key in texts:
                continue
            try:
                root = ET.parse(xml_file).getroot()
                raw = _ca_xml_text(root)
                if raw:
                    texts[key] = raw
            except Exception as exc:
                print(f'Warning: {xml_file.name}: {exc}')
    return texts

_SKIP_TAGS = {'metadata', 'dublinCore', 'form', 'distribution-code', 'congress', 'session', 'current-chamber', 'action', 'action-date', 'action-desc', 'legis-type', 'enum', 'external-xref', 'quote'}
_PREFIX_TO_TYPE = {'HB': 'hr', 'SB': 's', 'HR': 'hres', 'SR': 'sres', 'HJR': 'hjres', 'SJR': 'sjres', 'HCR': 'hconres', 'SCR': 'sconres'}
_TYPE_TO_PREFIX = {v: k for k, v in _PREFIX_TO_TYPE.items()}

def _xml_text(el) -> str:
    if el.tag in _SKIP_TAGS:
        return ''
    parts = [el.text or '']
    for child in el:
        parts.append(_xml_text(child))
        parts.append(child.tail or '')
    return ' '.join(p.strip() for p in parts if p.strip())

def load_us_texts(us_txt_dir: Path) -> dict:
    texts = {}
    for zip_path in sorted(us_txt_dir.glob('*.zip')):
        parts = zip_path.stem.split('_')
        congress = parts[0]
        xml_type = parts[-1]
        legiscan_prefix = _TYPE_TO_PREFIX.get(xml_type)
        if legiscan_prefix is None:
            continue
        try:
            with zipfile.ZipFile(zip_path) as zf:
                for name in zf.namelist():
                    if not name.endswith('.xml'):
                        continue
                    match = re.match(rf'BILLS-{congress}{xml_type}(\d+)', name, re.IGNORECASE)
                    if not match:
                        continue
                    key = f'{congress}/{legiscan_prefix}{int(match.group(1))}'
                    try:
                        root = ET.fromstring(zf.read(name))
                        body = root.find('./legis-body')
                        if body is not None:
                            texts[key] = _xml_text(body)
                    except Exception:
                        pass
        except Exception as exc:
            print(f'Warning: {zip_path.name}: {exc}')
    return texts

ca_texts = load_canada_texts(CA_TXT)
us_texts = load_us_texts(US_TXT)

bills['full_text'] = ''
mask_ca = bills['source'] == 'canada'
mask_us = bills['source'] == 'us'
bills.loc[mask_ca, 'full_text'] = bills[mask_ca].apply(lambda r: ca_texts.get(f"{r['session']}/{r['bill_number']}", ''), axis=1)
bills.loc[mask_us, 'full_text'] = bills[mask_us].apply(lambda r: us_texts.get(f"{r['session']}/{r['bill_number']}", ''), axis=1)

print(f"Loaded {len(bills):,} bills")
print(f"Bills with full text: {(bills['full_text'].str.len() > 0).sum():,} / {len(bills):,}")
print(bills.groupby('source')['full_text'].apply(lambda s: (s.str.len() > 0).mean()).round(3))

Loaded 120,339 rows from bills.csv
Sources: {'us': 113231, 'canada': 7108}
Bills with full text: 81,631 / 120,339
source
canada    0.860
us        0.667
Name: full_text, dtype: float64


## 2. Country Splits and Transfer-Safe Metadata

Use chronological splits within each country, then fit metadata encoders on the training country only so transfer evaluation does not leak vocabulary or normalization statistics.

In [ ]:
@dataclass
class CountrySplit:
    train_df: pd.DataFrame
    val_df: pd.DataFrame
    test_df: pd.DataFrame


def make_country_split(df: pd.DataFrame, source: str, train_frac: float = TRAIN_FRAC, val_frac: float = VAL_FRAC) -> CountrySplit:
    country_df = df[df['source'] == source].copy()
    country_df = country_df.sort_values(['introduced_date', '_row_id']).reset_index(drop=True)
    if len(country_df) < 3:
        raise ValueError(f'Not enough rows for {source} split')

    n_total = len(country_df)
    n_train = max(1, int(n_total * train_frac))
    n_val = max(1, int(n_total * val_frac))
    if n_train + n_val >= n_total:
        n_train = max(1, n_total - 2)
        n_val = 1

    n_test = n_total - n_train - n_val
    if n_test < 1:
        n_test = 1
        n_train = max(1, n_train - 1)

    train_df = country_df.iloc[:n_train].copy()
    val_df = country_df.iloc[n_train:n_train + n_val].copy()
    test_df = country_df.iloc[n_train + n_val:].copy()

    if len(test_df) == 0:
        test_df = country_df.iloc[-1:].copy()
        val_df = country_df.iloc[n_train:-1].copy() if n_total - n_train > 1 else country_df.iloc[n_train:n_train + 1].copy()

    return CountrySplit(train_df=train_df, val_df=val_df, test_df=test_df)


def build_country_splits(df: pd.DataFrame) -> dict[str, CountrySplit]:
    return {
        'canada': make_country_split(df, 'canada'),
        'us': make_country_split(df, 'us'),
    }


def fit_feature_state(train_df: pd.DataFrame) -> dict:
    num_means = train_df[NUMERIC_COLS].mean(numeric_only=True)
    num_stds = train_df[NUMERIC_COLS].std(numeric_only=True).replace(0, 1).fillna(1)

    cat_maps = {}
    cat_cardinalities = []
    for col in CATEGORICAL_COLS:
        values = ['<PAD>', '<UNK>'] + sorted(v for v in train_df[col].fillna('').astype(str).unique().tolist() if v)
        mapping = {v: i for i, v in enumerate(values)}
        cat_maps[col] = mapping
        cat_cardinalities.append(len(values))

    return {
        'num_means': num_means,
        'num_stds': num_stds,
        'cat_maps': cat_maps,
        'cat_cardinalities': cat_cardinalities,
    }


def transform_metadata(df: pd.DataFrame, feature_state: dict) -> tuple[np.ndarray, np.ndarray]:
    cat_arrays = []
    for col in CATEGORICAL_COLS:
        mapping = feature_state['cat_maps'][col]
        cat_arrays.append(df[col].fillna('').astype(str).map(lambda x, m=mapping: m.get(x, 1)).to_numpy())
    cat_x = np.stack(cat_arrays, axis=1)
    num_x = ((df[NUMERIC_COLS].fillna(feature_state['num_means']) - feature_state['num_means']) / feature_state['num_stds']).astype('float32').to_numpy()
    return cat_x.astype('int64'), num_x.astype('float32')


country_splits = build_country_splits(bills)
for country, split in country_splits.items():
    print(f"{country.upper()}: train={len(split.train_df):,}, val={len(split.val_df):,}, test={len(split.test_df):,}")

Categorical array shape: (120339, 4)
Numeric array shape: (120339, 7)
Label positive rate: 0.0287


## 3. Tokenization, Chunking, and Dataset

Bills can exceed BERT’s token limit, so the notebook uses overlapping chunks and pools chunk-level CLS embeddings into one document vector.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def chunk_text(text: str, tokenizer, max_length: int = MAX_LENGTH, stride: int = CHUNK_STRIDE):
    enc = tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        padding='max_length',
        return_attention_mask=True,
        return_tensors='pt',
    )
    return enc['input_ids'], enc['attention_mask']


class BillLateFusionDataset(Dataset):
    def __init__(self, df: pd.DataFrame, texts: list[str], cat_x: np.ndarray, num_x: np.ndarray, labels: np.ndarray):
        self.df = df.reset_index(drop=True)
        self.texts = texts
        self.cat_x = cat_x
        self.num_x = num_x
        self.labels = labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            'text': self.texts[idx],
            'categorical': torch.tensor(self.cat_x[idx], dtype=torch.long),
            'numerical': torch.tensor(self.num_x[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32),
        }


def collate_batch(batch):
    texts = [item['text'] for item in batch]
    cats = torch.stack([item['categorical'] for item in batch])
    nums = torch.stack([item['numerical'] for item in batch])
    labels = torch.stack([item['label'] for item in batch])

    chunked_ids = []
    chunked_masks = []
    max_chunks = 1
    for text in texts:
        input_ids, attention_mask = chunk_text(text, tokenizer)
        input_ids = input_ids[:MAX_CHUNKS]
        attention_mask = attention_mask[:MAX_CHUNKS]
        chunked_ids.append(input_ids)
        chunked_masks.append(attention_mask)
        max_chunks = max(max_chunks, input_ids.size(0))

    seq_len = chunked_ids[0].size(-1)
    padded_ids = []
    padded_masks = []
    for input_ids, attention_mask in zip(chunked_ids, chunked_masks):
        if input_ids.size(0) < max_chunks:
            pad_shape = (max_chunks - input_ids.size(0), seq_len)
            id_pad = torch.zeros(pad_shape, dtype=torch.long)
            mask_pad = torch.zeros(pad_shape, dtype=torch.long)
            input_ids = torch.cat([input_ids, id_pad], dim=0)
            attention_mask = torch.cat([attention_mask, mask_pad], dim=0)
        padded_ids.append(input_ids)
        padded_masks.append(attention_mask)

    return {
        'input_ids': torch.stack(padded_ids),
        'attention_mask': torch.stack(padded_masks),
        'categorical': cats,
        'numerical': nums,
        'labels': labels,
    }


def make_bill_dataset(df: pd.DataFrame, feature_state: dict) -> BillLateFusionDataset:
    cat_x, num_x = transform_metadata(df, feature_state)
    texts = df['full_text'].fillna('').astype(str).tolist()
    labels = df[TARGET_COL].to_numpy(dtype='float32')
    return BillLateFusionDataset(df, texts, cat_x, num_x, labels)


sample_ds = make_bill_dataset(country_splits['canada'].train_df.head(2).copy(), fit_feature_state(country_splits['canada'].train_df))
sample = collate_batch([sample_ds[0], sample_ds[1]])
print({k: tuple(v.shape) for k, v in sample.items()})

{'input_ids': (2, 1, 256), 'attention_mask': (2, 1, 256), 'categorical': (2, 4), 'numerical': (2, 7), 'labels': (2,)}


## 4. Late Fusion Model

The model uses a frozen BERT encoder for text, small embeddings for the transferable categorical metadata, and a linear projection for numeric metadata before the fused MLP head.

In [ ]:
class LateFusionBillClassifier(nn.Module):
    def __init__(
        self,
        model_name: str,
        cat_cardinalities: list[int],
        num_numeric: int,
        bert_hidden_size: int = 768,
        cat_embed_dim: int = 16,
        num_proj_dim: int = 32,
        mlp_hidden_dims: list[int] | None = None,
        dropout: float = 0.2,
        freeze_bert: bool = True,
    ):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        bert_hidden_size = getattr(self.bert.config, 'hidden_size', bert_hidden_size)
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        self.cat_embeddings = nn.ModuleList([nn.Embedding(cardinality, cat_embed_dim) for cardinality in cat_cardinalities])
        self.num_projection = nn.Linear(num_numeric, num_proj_dim)

        fusion_dim = bert_hidden_size + (len(cat_cardinalities) * cat_embed_dim) + num_proj_dim
        mlp_hidden_dims = mlp_hidden_dims or [256, 128]

        layers = []
        in_dim = fusion_dim
        for hidden_dim in mlp_hidden_dims:
            layers.extend([
                nn.Linear(in_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1))
        self.classifier = nn.Sequential(*layers)

    def encode_text(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        batch_size, num_chunks, seq_len = input_ids.shape
        flat_ids = input_ids.view(batch_size * num_chunks, seq_len)
        flat_mask = attention_mask.view(batch_size * num_chunks, seq_len)
        outputs = self.bert(input_ids=flat_ids, attention_mask=flat_mask)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        cls_embeddings = cls_embeddings.view(batch_size, num_chunks, -1)
        return cls_embeddings.mean(dim=1)

    def encode_tabular(self, categorical: torch.Tensor, numerical: torch.Tensor) -> torch.Tensor:
        cat_parts = []
        for idx, embedding in enumerate(self.cat_embeddings):
            cat_parts.append(embedding(categorical[:, idx]))
        cat_vector = torch.cat(cat_parts, dim=1) if cat_parts else categorical.new_zeros((categorical.size(0), 0), dtype=torch.float32)
        num_vector = self.num_projection(numerical)
        return torch.cat([cat_vector, num_vector], dim=1)

    def forward(self, input_ids, attention_mask, categorical, numerical):
        text_vector = self.encode_text(input_ids, attention_mask)
        tabular_vector = self.encode_tabular(categorical, numerical)
        fused = torch.cat([text_vector, tabular_vector], dim=1)
        return self.classifier(fused).squeeze(-1)

print('LateFusionBillClassifier defined.')

Train batches: 48136
Val batches: 12034


## 5. Loss Function and Training Utilities

Use weighted BCE as the default objective and keep focal loss available as an option for class imbalance.

In [8]:
class LateFusionBillClassifier(nn.Module):
    def __init__(
        self,
        model_name: str,
        cat_cardinalities: list[int],
        num_numeric: int,
        bert_hidden_size: int = 768,
        cat_embed_dim: int = 16,
        num_proj_dim: int = 64,
        mlp_hidden_dims: list[int] | None = None,
        dropout: float = 0.2,
        freeze_bert: bool = True,
    ):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        bert_hidden_size = getattr(self.bert.config, 'hidden_size', bert_hidden_size)
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        self.cat_embeddings = nn.ModuleList([nn.Embedding(cardinality, cat_embed_dim) for cardinality in cat_cardinalities])
        self.num_projection = nn.Linear(num_numeric, num_proj_dim)

        fusion_dim = bert_hidden_size + (len(cat_cardinalities) * cat_embed_dim) + num_proj_dim
        mlp_hidden_dims = mlp_hidden_dims or [256, 128]

        layers = []
        in_dim = fusion_dim
        for hidden_dim in mlp_hidden_dims:
            layers.extend([
                nn.Linear(in_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1))
        self.classifier = nn.Sequential(*layers)

    def encode_text(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        batch_size, num_chunks, seq_len = input_ids.shape
        flat_ids = input_ids.view(batch_size * num_chunks, seq_len)
        flat_mask = attention_mask.view(batch_size * num_chunks, seq_len)
        outputs = self.bert(input_ids=flat_ids, attention_mask=flat_mask)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        cls_embeddings = cls_embeddings.view(batch_size, num_chunks, -1)
        doc_embeddings = cls_embeddings.mean(dim=1)
        return doc_embeddings

    def encode_tabular(self, categorical: torch.Tensor, numerical: torch.Tensor) -> torch.Tensor:
        cat_parts = []
        for idx, embedding in enumerate(self.cat_embeddings):
            cat_parts.append(embedding(categorical[:, idx]))
        cat_vector = torch.cat(cat_parts, dim=1) if cat_parts else categorical.new_zeros((categorical.size(0), 0), dtype=torch.float32)
        num_vector = self.num_projection(numerical)
        return torch.cat([cat_vector, num_vector], dim=1)

    def forward(self, input_ids, attention_mask, categorical, numerical):
        text_vector = self.encode_text(input_ids, attention_mask)
        tabular_vector = self.encode_tabular(categorical, numerical)
        fused = torch.cat([text_vector, tabular_vector], dim=1)
        logits = self.classifier(fused).squeeze(-1)
        return logits

model = LateFusionBillClassifier(
    model_name=MODEL_NAME,
    cat_cardinalities=cat_cardinalities,
    num_numeric=len(NUMERIC_COLS),
    freeze_bert=True,
)
print(model.__class__.__name__)

c:\Users\tamze\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


LateFusionBillClassifier


## 9. Final Transfer Report

Combine the Canada-trained and US-trained results into one report and summarize the most important tokens and metadata features for each model.

In [ ]:
def make_pos_weight(labels: np.ndarray) -> torch.Tensor:
    positives = max(float(labels.sum()), 1.0)
    negatives = max(float(len(labels) - labels.sum()), 1.0)
    return torch.tensor([negatives / positives], dtype=torch.float32)


class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        loss = alpha_t * (1 - pt).pow(self.gamma) * bce
        if self.reduction == 'mean':
            return loss.mean()
        if self.reduction == 'sum':
            return loss.sum()
        return loss


def predict_loader(model, loader, device):
    model.eval()
    all_logits = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            logits = model(
                batch['input_ids'].to(device),
                batch['attention_mask'].to(device),
                batch['categorical'].to(device),
                batch['numerical'].to(device),
            )
            all_logits.append(logits.detach().cpu())
            all_labels.append(batch['labels'].detach().cpu())
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probs = 1.0 / (1.0 + np.exp(-logits))
    return labels, probs, logits


def compute_metrics(y_true, y_prob, threshold: float = 0.5) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'pr_auc': float(average_precision_score(y_true, y_prob)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
    }


def best_f1_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    if len(thresholds) == 0:
        return 0.5
    f1_vals = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    return float(thresholds[int(np.nanargmax(f1_vals))])


def train_one_epoch(model, loader, optimizer, criterion, device, grad_accum_steps=1, use_amp=False, scaler=None, show_progress=False, progress_desc='Train'):
    model.train()
    total_loss = 0.0
    total_examples = 0
    optimizer.zero_grad(set_to_none=True)
    iterator = tqdm(loader, desc=progress_desc, leave=False, dynamic_ncols=True) if show_progress else loader
    for step, batch in enumerate(iterator, start=1):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        categorical = batch['categorical'].to(device)
        numerical = batch['numerical'].to(device)
        labels = batch['labels'].to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            logits = model(input_ids, attention_mask, categorical, numerical)
            raw_loss = criterion(logits, labels)
            loss = raw_loss / grad_accum_steps
        if use_amp and scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        if step % grad_accum_steps == 0 or step == len(loader):
            if use_amp and scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        total_loss += raw_loss.item() * labels.size(0)
        total_examples += labels.size(0)
        if show_progress and hasattr(iterator, 'set_postfix'):
            iterator.set_postfix(loss=f'{raw_loss.item():.4f}')
    return total_loss / max(total_examples, 1)

print('Loss and training utilities defined.')

pos_weight = 33.861


## 6. Evaluation and Transfer Helpers

These helpers evaluate a trained country model on its own test set and on the other country’s test set, then compute token and metadata importance from the held-out data.

In [ ]:
def make_dataset(df: pd.DataFrame, feature_state: dict) -> BillLateFusionDataset:
    cat_x, num_x = transform_metadata(df, feature_state)
    texts = df['full_text'].fillna('').astype(str).tolist()
    labels = df[TARGET_COL].to_numpy(dtype='float32')
    return BillLateFusionDataset(df.reset_index(drop=True), texts, cat_x, num_x, labels)


def make_loader(df: pd.DataFrame, feature_state: dict, shuffle: bool = False) -> torch.utils.data.DataLoader:
    return torch.utils.data.DataLoader(
        make_dataset(df, feature_state),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_batch,
        pin_memory=(DEVICE.type == 'cuda'),
    )


def evaluate_model(model, loader, device, threshold: float) -> dict:
    y_true, y_prob, _ = predict_loader(model, loader, device)
    return compute_metrics(y_true, y_prob, threshold=threshold)


def train_country_model(train_df: pd.DataFrame, val_df: pd.DataFrame, train_source: str) -> tuple[LateFusionBillClassifier, dict, float, pd.DataFrame]:
    feature_state = fit_feature_state(train_df)
    train_loader = make_loader(train_df, feature_state, shuffle=True)
    val_loader = make_loader(val_df, feature_state, shuffle=False)
    model = LateFusionBillClassifier(
        model_name=MODEL_NAME,
        cat_cardinalities=feature_state['cat_cardinalities'],
        num_numeric=len(NUMERIC_COLS),
        freeze_bert=True,
    ).to(DEVICE)
    pos_weight = make_pos_weight(train_df[TARGET_COL].to_numpy(dtype='float32')).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LR)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))
    history = []
    grad_accum_steps = max(1, 8 // max(1, BATCH_SIZE))
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, DEVICE,
            grad_accum_steps=grad_accum_steps,
            use_amp=(DEVICE.type == 'cuda'),
            scaler=scaler,
            show_progress=True,
            progress_desc=f'{train_source.title()} train {epoch}/{EPOCHS}',
        )
        val_labels, val_probs, _ = predict_loader(model, val_loader, DEVICE)
        val_thr = best_f1_threshold(val_labels, val_probs)
        val_metrics = compute_metrics(val_labels, val_probs, threshold=val_thr)
        history.append({'epoch': epoch, 'train_loss': train_loss, **{k: v for k, v in val_metrics.items() if k != 'confusion_matrix'}})
        print(f"Epoch {epoch}/{EPOCHS} | train_loss={train_loss:.4f} | val_pr_auc={val_metrics['pr_auc']:.4f} | val_f1={val_metrics['f1']:.4f}")
    history_df = pd.DataFrame(history)
    final_val_labels, final_val_probs, _ = predict_loader(model, val_loader, DEVICE)
    threshold = best_f1_threshold(final_val_labels, final_val_probs)
    return model, feature_state, threshold, history_df


def top_token_importance_for_sample(model, dataset, sample_index, tokenizer, device, top_k=12):
    item = dataset[sample_index]
    batch = collate_batch([item])
    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']
    categorical = batch['categorical']
    numerical = batch['numerical']
    logits = model(
        input_ids.to(device),
        attention_mask.to(device),
        categorical.to(device),
        numerical.to(device),
    )
    base_prob = float(torch.sigmoid(logits).squeeze().detach().cpu().item())
    ids = input_ids[0, 0].clone()
    mask = attention_mask[0, 0].clone()
    valid_positions = torch.where(mask == 1)[0].tolist()
    special_ids = {tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id}
    replace_id = tokenizer.mask_token_id if tokenizer.mask_token_id is not None else tokenizer.unk_token_id
    rows = []
    for pos in valid_positions:
        tok_id = int(ids[pos].item())
        if tok_id in special_ids:
            continue
        perturbed_ids = input_ids.clone()
        perturbed_ids[0, 0, pos] = replace_id
        pert_logits = model(
            perturbed_ids.to(device),
            attention_mask.to(device),
            categorical.to(device),
            numerical.to(device),
        )
        pert_prob = float(torch.sigmoid(pert_logits).squeeze().detach().cpu().item())
        token_str = tokenizer.convert_ids_to_tokens([tok_id])[0]
        rows.append({'position': pos, 'token': token_str, 'delta_prob': base_prob - pert_prob})
    imp = pd.DataFrame(rows).sort_values('delta_prob', ascending=False)
    helpful = imp.head(top_k).reset_index(drop=True)
    harmful = imp.tail(top_k).sort_values('delta_prob', ascending=True).reset_index(drop=True)
    return base_prob, helpful, harmful


def permutation_importance_tabular(model, eval_df: pd.DataFrame, feature_state: dict, device, metric_fn, n_repeats: int = 1):
    base_cat, base_num = transform_metadata(eval_df, feature_state)
    base_dataset = make_dataset(eval_df, feature_state)
    base_loader = torch.utils.data.DataLoader(base_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
    y_true, y_prob, _ = predict_loader(model, base_loader, device)
    base_metric = metric_fn(y_true, y_prob)
    rng = np.random.default_rng(SEED)
    rows = []
    for j, col in enumerate(NUMERIC_COLS):
        scores = []
        for _ in range(n_repeats):
            pert_num = base_num.copy()
            rng.shuffle(pert_num[:, j])
            pert_dataset = BillLateFusionDataset(eval_df.reset_index(drop=True), eval_df['full_text'].fillna('').astype(str).tolist(), base_cat, pert_num, eval_df[TARGET_COL].to_numpy(dtype='float32'))
            pert_loader = torch.utils.data.DataLoader(pert_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
            yt, yp, _ = predict_loader(model, pert_loader, device)
            scores.append(metric_fn(yt, yp))
        rows.append({'feature': col, 'type': 'numeric', 'metric_drop': base_metric - float(np.mean(scores))})
    for j, col in enumerate(CATEGORICAL_COLS):
        scores = []
        for _ in range(n_repeats):
            pert_cat = base_cat.copy()
            rng.shuffle(pert_cat[:, j])
            pert_dataset = BillLateFusionDataset(eval_df.reset_index(drop=True), eval_df['full_text'].fillna('').astype(str).tolist(), pert_cat, base_num, eval_df[TARGET_COL].to_numpy(dtype='float32'))
            pert_loader = torch.utils.data.DataLoader(pert_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
            yt, yp, _ = predict_loader(model, pert_loader, device)
            scores.append(metric_fn(yt, yp))
        rows.append({'feature': col, 'type': 'categorical', 'metric_drop': base_metric - float(np.mean(scores))})
    return base_metric, pd.DataFrame(rows).sort_values('metric_drop', ascending=False).reset_index(drop=True)


def run_country_transfer_experiment(train_source: str, country_splits: dict[str, CountrySplit]) -> dict:
    train_split = country_splits[train_source]
    other_source = 'us' if train_source == 'canada' else 'canada'
    other_split = country_splits[other_source]

    model, feature_state, threshold, history_df = train_country_model(train_split.train_df, train_split.val_df, train_source)

    in_test_df = train_split.test_df.copy()
    out_test_df = other_split.test_df.copy()

    in_test_loader = make_loader(in_test_df, feature_state, shuffle=False)
    out_test_loader = make_loader(out_test_df, feature_state, shuffle=False)

    in_metrics = evaluate_model(model, in_test_loader, DEVICE, threshold)
    out_metrics = evaluate_model(model, out_test_loader, DEVICE, threshold)

    in_dataset = make_dataset(in_test_df, feature_state)
    sample_idx = 0
    base_prob, top_helpful, top_harmful = top_token_importance_for_sample(model, in_dataset, sample_idx, tokenizer, DEVICE, top_k=12)
    base_pr_auc, feature_importance = permutation_importance_tabular(
        model,
        in_test_df,
        feature_state,
        DEVICE,
        metric_fn=lambda y, p: average_precision_score(y, p),
    )

    report_rows = [
        {
            'train_country': train_source,
            'test_country': train_source,
            'setting': 'in_country_test',
            'threshold': threshold,
            **in_metrics,
        },
        {
            'train_country': train_source,
            'test_country': other_source,
            'setting': 'transfer_test',
            'threshold': threshold,
            **out_metrics,
        },
    ]

    return {
        'train_country': train_source,
        'other_country': other_source,
        'model': model,
        'feature_state': feature_state,
        'threshold': threshold,
        'history': history_df,
        'report': pd.DataFrame(report_rows),
        'token_importance': {'base_prob': base_prob, 'helpful': top_helpful, 'harmful': top_harmful},
        'feature_importance': pd.DataFrame(feature_importance),
        'base_pr_auc': base_pr_auc,
    }

print('Evaluation and transfer helpers defined.')

Smoke test logits shape: (2,)


## 7. Canada-Only Training Run

Train on Canadian bills only, evaluate on Canadian test bills, and then transfer the same model to the US test set.

In [ ]:
print('Running Canada-only transfer experiment...')
canada_run = run_country_transfer_experiment('canada', country_splits)

print('\nCanada training history:')
display(canada_run['history'])

print('\nCanada transfer report:')
display(canada_run['report'])

print('\nCanada token importance on Canadian test sample:')
print(f"Base probability: {canada_run['token_importance']['base_prob']:.4f}")
display(canada_run['token_importance']['helpful'])
display(canada_run['token_importance']['harmful'])

print('\nCanada metadata importance:')
display(canada_run['feature_importance'])

Evaluation and interpretability helpers ready.


## 8. US-Only Training Run

Train on US bills only, evaluate on US test bills, and then transfer the same model to the Canadian test set.

In [ ]:
print('Running US-only transfer experiment...')
us_run = run_country_transfer_experiment('us', country_splits)

print('\nUS training history:')
display(us_run['history'])

print('\nUS transfer report:')
display(us_run['report'])

print('\nUS token importance on US test sample:')
print(f"Base probability: {us_run['token_importance']['base_prob']:.4f}")
display(us_run['token_importance']['helpful'])
display(us_run['token_importance']['harmful'])

print('\nUS metadata importance:')
display(us_run['feature_importance'])

final_report = pd.concat([canada_run['report'], us_run['report']], ignore_index=True)
final_report = final_report[[
    'train_country',
    'test_country',
    'setting',
    'threshold',
    'accuracy',
    'balanced_accuracy',
    'pr_auc',
    'roc_auc',
    'f1',
    'precision',
    'recall',
]]
print('\nCombined transfer report:')
display(final_report)

report_path = NORM / 'late_fusion_transfer_report.json'
with open(report_path, 'w', encoding='utf-8') as fp:
    json.dump(final_report.to_dict(orient='records'), fp, indent=2)
print(f'Saved transfer report to {report_path}')

print('\nTop metadata features by PR-AUC drop (Canada-trained model):')
display(canada_run['feature_importance'].head(10))

print('\nTop metadata features by PR-AUC drop (US-trained model):')
display(us_run['feature_importance'].head(10))

print('\nCanada-trained model token importance:')
display(canada_run['token_importance']['helpful'])
display(canada_run['token_importance']['harmful'])

print('\nUS-trained model token importance:')
display(us_run['token_importance']['helpful'])
display(us_run['token_importance']['harmful'])

Using device=cuda, batch_size=1, grad_accum_steps=8, amp=True


Epochs:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 1/5:   0%|          | 0/96271 [00:00<?, ?it/s]

KeyboardInterrupt: 